# SAPN2022 Aggregate Curtailment Plots

This notebook builds daily aggregate curtailment pie charts from `curtailment_sapn2022_5m.parquet`.
It is organised as simple plot sections so the same pattern can be reused as more plots are added.


In [ ]:
from __future__ import annotations

from itertools import cycle, islice
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl


def find_curtailment_scripts_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        direct = candidate / "SAPN2022_Analysis" / "curtailment" / "scripts"
        if (direct / "plot_aggregates.ipynb").exists():
            return direct
        if (candidate / "plot_aggregates.ipynb").exists() and candidate.name == "scripts":
            return candidate
    raise RuntimeError(
        "Could not locate SAPN2022_Analysis/curtailment/scripts from the current working directory."
    )
    


In [ ]:
# Repo-contained outputs are resolved relative to the tracked curtailment folder
# so this notebook does not depend on a hard-coded machine path.
SCRIPTS_DIR = find_curtailment_scripts_dir(Path.cwd().resolve())
PROJECT_ROOT = SCRIPTS_DIR.parent

# Default parquet file to read.
DEFAULT_CURTAILMENT_SUMMARY = (
    PROJECT_ROOT / "outputs" / "curtailed_estimates_5m" / "curtailment_sapn2022_5m.parquet"
)

# Default folder where plot images can be saved.
DEFAULT_folder_SAVE_PATH = PROJECT_ROOT / "outputs" / "plots" / "aggregated"

GRAFANA_CLASSIC_COLORS = (
    "#7EB26D",
    "#EAB839",
    "#6ED0E0",
    "#EF843C",
    "#E24D42",
    "#1F78C1",
)
TEXT_COLOR = "#111827"

DEFAULT_CURTAILMENT_SUMMARY, DEFAULT_folder_SAVE_PATH
    


In [ ]:
def _slice_colors(count: int) -> list[str]:
    return list(islice(cycle(GRAFANA_CLASSIC_COLORS), count))


def _format_pct_label(pct: float) -> str:
    return f"{pct:.0f}%" if pct > 0 else ""


## Raw summed 5-minute kW pie

This function groups the parquet by day and plots each day's share of the raw summed 5-minute curtailed power.
Pass the summary path, save folder, and filename explicitly so the inputs stay visible in the notebook.


In [ ]:
def plot_daily_curtailment_pie(
    summary_path: Path,
    save_folder: Path,
    filename: str,
    save: bool = False,
):
    """Plot daily curtailed power share using the raw summed 5-minute kW values."""
    summary_path = Path(summary_path)
    save_folder = Path(save_folder)

    # Read the curtailment summary parquet.
    curtailment_df = pl.read_parquet(summary_path)

    # Stop early if the parquet has no rows.
    if curtailment_df.is_empty():
        raise ValueError(f"Curtailment summary parquet is empty: {summary_path}")

    # Group the raw curtailed power by calendar day.
    daily_totals = (
        curtailment_df
        .group_by(["year", "month", "day"])
        .agg(pl.col("curtailment_sapn2022_sum").sum().alias("daily_raw_sum"))
        .filter(pl.col("daily_raw_sum") > 0)
        .sort(["year", "month", "day"])
    )

    # Stop if no positive daily totals remain after grouping.
    if daily_totals.is_empty():
        raise ValueError(
            "Curtailment summary parquet contains no positive daily totals after aggregation: "
            f"{summary_path}"
        )

    # Use the aggregated daily total to calculate each pie slice share.
    total_raw_sum = daily_totals["daily_raw_sum"].sum()
    if total_raw_sum <= 0:
        raise ValueError(
            "Curtailment summary parquet contains a non-positive aggregated total: "
            f"{summary_path}"
        )

    # Build a readable date label for the legend and calculate percent share.
    plot_df = daily_totals.with_columns([
        pl.date(pl.col("year"), pl.col("month"), pl.col("day"))
        .dt.strftime("%Y-%m-%d")
        .alias("date_label"),
        (pl.col("daily_raw_sum") / pl.lit(total_raw_sum) * 100.0).alias("pct_share"),
    ]).sort("daily_raw_sum", descending=True)

    date_labels = plot_df["date_label"].to_list()
    values = plot_df["daily_raw_sum"].to_list()
    legend_labels = [
        f"{date_label}  Value: {value:,.1f} kW"
        for date_label, value in zip(date_labels, values, strict=True)
    ]

    # Draw the pie chart.
    fig, ax = plt.subplots(figsize=(12.0, 6.5), dpi=300)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    wedges, _, autotexts = ax.pie(
        values,
        colors=_slice_colors(len(values)),
        autopct=_format_pct_label,
        startangle=90,
        counterclock=False,
        wedgeprops={"edgecolor": "white", "linewidth": 1.0},
        textprops={"color": TEXT_COLOR, "fontsize": 11},
    )

    for autotext in autotexts:
        autotext.set_color(TEXT_COLOR)
        autotext.set_fontsize(11)

    ax.set_title(
        "SAPN2022 curtailed power share by day (raw summed 5-minute kW)",
        fontsize=14,
        loc="left",
        pad=16,
    )
    ax.axis("equal")
    ax.legend(
        wedges,
        legend_labels,
        loc="center left",
        bbox_to_anchor=(1.0, 0.5),
        frameon=False,
        fontsize=11,
        handlelength=1.2,
        labelspacing=1.0,
        borderaxespad=0.0,
    )
    fig.subplots_adjust(left=0.06, right=0.74, top=0.90, bottom=0.08)

    # Save the figure only when requested.
    if save:
        output_path = save_folder / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return fig, ax


In [ ]:
# Name the output file here so it is easy to change before calling the function.
raw_filename = "curtailment_sapn2022_daily_pie_raw_sum.png"

fig_raw, ax_raw = plot_daily_curtailment_pie(
    summary_path=DEFAULT_CURTAILMENT_SUMMARY,
    save_folder=DEFAULT_folder_SAVE_PATH,
    filename=raw_filename,
    save=False,
)
plt.show()


## kWh pie

This function uses the same daily grouping, then converts the grouped raw total into `kWh` with `5 / 60` before plotting.
Again, keep the summary path, save folder, and filename visible at the call site.


In [ ]:
def plot_daily_curtailment_pie_kwh(
    summary_path: Path,
    save_folder: Path,
    filename: str,
    save: bool = False,
):
    """Plot daily curtailed energy share using kWh converted from 5-minute values."""
    summary_path = Path(summary_path)
    save_folder = Path(save_folder)

    # Read the curtailment summary parquet.
    curtailment_df = pl.read_parquet(summary_path)

    # Stop early if the parquet has no rows.
    if curtailment_df.is_empty():
        raise ValueError(f"Curtailment summary parquet is empty: {summary_path}")

    # Group the raw curtailed power by calendar day.
    daily_totals = (
        curtailment_df
        .group_by(["year", "month", "day"])
        .agg(pl.col("curtailment_sapn2022_sum").sum().alias("daily_raw_sum"))
        .filter(pl.col("daily_raw_sum") > 0)
        .sort(["year", "month", "day"])
    )

    # Stop if no positive daily totals remain after grouping.
    if daily_totals.is_empty():
        raise ValueError(
            "Curtailment summary parquet contains no positive daily totals after aggregation: "
            f"{summary_path}"
        )

    # Use the aggregated daily total to calculate each pie slice share.
    total_raw_sum = daily_totals["daily_raw_sum"].sum()
    if total_raw_sum <= 0:
        raise ValueError(
            "Curtailment summary parquet contains a non-positive aggregated total: "
            f"{summary_path}"
        )

    # Build a readable date label, convert the grouped total to kWh, and calculate percent share.
    plot_df = daily_totals.with_columns([
        pl.date(pl.col("year"), pl.col("month"), pl.col("day"))
        .dt.strftime("%Y-%m-%d")
        .alias("date_label"),
        (pl.col("daily_raw_sum") * (5.0 / 60.0)).alias("daily_kwh"),
        (pl.col("daily_raw_sum") / pl.lit(total_raw_sum) * 100.0).alias("pct_share"),
    ]).sort("daily_kwh", descending=True)

    date_labels = plot_df["date_label"].to_list()
    values = plot_df["daily_kwh"].to_list()
    legend_labels = [
        f"{date_label}  Value: {value:,.1f} kWh"
        for date_label, value in zip(date_labels, values, strict=True)
    ]

    # Draw the pie chart.
    fig, ax = plt.subplots(figsize=(12.0, 6.5), dpi=300)
    fig.patch.set_facecolor("white")
    ax.set_facecolor("white")

    wedges, _, autotexts = ax.pie(
        values,
        colors=_slice_colors(len(values)),
        autopct=_format_pct_label,
        startangle=90,
        counterclock=False,
        wedgeprops={"edgecolor": "white", "linewidth": 1.0},
        textprops={"color": TEXT_COLOR, "fontsize": 11},
    )

    for autotext in autotexts:
        autotext.set_color(TEXT_COLOR)
        autotext.set_fontsize(11)

    ax.set_title(
        "SAPN2022 curtailed energy share by day (kWh)",
        fontsize=14,
        loc="left",
        pad=16,
    )
    ax.axis("equal")
    ax.legend(
        wedges,
        legend_labels,
        loc="center left",
        bbox_to_anchor=(1.0, 0.5),
        frameon=False,
        fontsize=11,
        handlelength=1.2,
        labelspacing=1.0,
        borderaxespad=0.0,
    )
    fig.subplots_adjust(left=0.06, right=0.74, top=0.90, bottom=0.08)

    # Save the figure only when requested.
    if save:
        output_path = save_folder / filename
        output_path.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output_path, dpi=300, bbox_inches="tight")

    return fig, ax


In [ ]:
# Name the output file here so it is easy to change before calling the function.
kwh_filename = "curtailment_sapn2022_daily_pie_kwh.png"

fig_kwh, ax_kwh = plot_daily_curtailment_pie_kwh(
    summary_path=DEFAULT_CURTAILMENT_SUMMARY,
    save_folder=DEFAULT_folder_SAVE_PATH,
    filename=kwh_filename,
    save=False,
)
plt.show()
